# MissingPatterns.jl — reading the holes in a table

[MissingPatterns.jl](https://github.com/dantebertuzzi/MissingPatterns.jl) answers a question
that comes before any model: *what is missing here, and is it missing at random?* It works on
anything [Tables.jl](https://github.com/JuliaData/Tables.jl) accepts — a `DataFrame`, a
`CSV.File`, a `NamedTuple` of vectors — and draws with Unicode and ANSI rather than a plotting
library, so the same diagnostic looks the same in a terminal, in CI logs, and here.

This notebook builds a table whose missingness has deliberate structure, and then finds that
structure with the package rather than by knowing where it was put.

Documentation: <https://dantebertuzzi.github.io/MissingPatterns.jl/stable>

**On Colab, pick the Julia runtime first:** *Runtime ▸ Change runtime type ▸ Julia*.

In [ ]:
using Pkg
Pkg.add(["MissingPatterns", "DataFrames"])

In [ ]:
using MissingPatterns, DataFrames, Dates, Random, Statistics

## A table with structure in its holes

A thousand records from an imaginary follow-up study. Three things are true of it by
construction, and none of them are visible in the table itself:

- **income** is refused more often in the south, and by the oldest respondents — missing *because
  of* values in other columns, not at random;
- **lab** and **weight** come from the same clinical visit, so they go missing together, and the
  visit was likelier to be skipped in the early years;
- **schooling** is never `missing` at all. It codes "ignored" as `99`, the way real
  administrative files do.

In [1]:
Random.seed!(20260911)
n = 1000

region    = rand(["north", "south", "east", "west"], n)
date      = Date(2019, 1, 1) .+ Day.(rand(0:2_190, n))
age       = rand(18:89, n)
schooling = rand([0, 1, 2, 3, 99], n)

income = Vector{Union{Missing,Int}}(round.(Int, 1500 .+ 120 .* (age .- 18) .+ 800 .* randn(n)))
lab    = Vector{Union{Missing,Float64}}(round.(3.5 .+ 0.9 .* randn(n); digits = 2))
weight = Vector{Union{Missing,Float64}}(round.(60 .+ 12 .* randn(n); digits = 1))

for i in 1:n
    # refusal to state income: more in the south, more among the oldest
    rand() < 0.10 + 0.25 * (region[i] == "south") + 0.20 * (age[i] > 70) && (income[i] = missing)

    # a skipped visit takes the lab result, and usually the weight, with it
    if rand() < 0.30 + 0.35 * (date[i] < Date(2021, 1, 1))
        lab[i] = missing
        rand() < 0.8 && (weight[i] = missing)
    end
end

df = DataFrame(; region, date, age, schooling, income, lab, weight)
first(df, 6)

Row,region,date,age,schooling,income,lab,weight
,String,Date,Int64,Int64,Int64?,Float64?,Float64?
1,west,2022-12-30,43,1,5596,1.5,95.2
2,east,2024-11-09,84,3,missing,missing,50.9
3,west,2022-12-19,42,0,4518,missing,67.9
4,east,2022-10-13,64,2,missing,missing,missing
5,south,2020-03-27,56,0,missing,missing,missing
6,east,2019-10-05,85,3,8935,missing,58.6


## Where the holes are

`missingreport` returns an object that renders as the terminal heatmap in a REPL and as an
HTML grid here, with a tooltip on every cell — the same numbers, drawn for whichever medium is
asking.

In [2]:
missingreport(df; order = :cluster)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ lab 44% ┃ we… 35% ┃ in… 20% ┃ reg… 0% ┃ date 0% ┃ age 0%  ┃ sch… 0% ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃  ▀▀▀▀▀  ┃
┗━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┛
 1000×7 → 46×7 (22×1/cell) ┊ missing 14.21% (995) ┊ present 85.79%

`plotmissing` is the terminal-only entry point, and prints the same thing as Unicode. Each cell
is a block of rows — 1000 rows do not fit on a screen, so they are compressed — and the glyph is
how much of that block is missing: `·` up to 5%, `░` up to 15%, `▒` up to 30%, `▓` up to 50%,
`█` above that.

`layout = :classic` draws one grid row per line. The default, `:auto`, would switch to the
`:compact` layout here, which packs two grid rows into each line as truecolor half-blocks — very
good in a terminal, less certain in whatever renders this notebook.

`order = :cluster` is doing real work. In table order the columns appear as they were written,
which scatters the ones that go missing together; clustering seriates the ϕ matrix of the masks
and puts them side by side, which is how `lab` and `weight` end up adjacent below.

In [3]:
plotmissing(df; order = :cluster, layout = :classic, max_rows = 25)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃   44%   ┃   35%   ┃   20%   ┃    0%   ┃    0%   ┃    0%   ┃    0%   ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃   lab   ┃  weig…  ┃  inco…  ┃  regi…  ┃  date   ┃   age   ┃  scho…  ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░

## Which columns go missing together

`missingpatterns` is `mice::md.pattern()` from R: one row per distinct combination of present
and absent columns, most frequent first. It is the honest answer to "how many complete cases do
I have", and to "what shape is the rest of the data".

In [4]:
missingpatterns(df)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃  regi…  ┃  date   ┃   age   ┃  scho…  ┃  inco…  ┃   lab   ┃  weig…  ┃    n    ┃    %    ┃  freq   ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃   436   ┃  43.6%  ┃ ███████ ┃
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃   284   ┃  28.4%  ┃ █████   ┃
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃  ░░░░░  ┃   122   ┃  12.2%  ┃ ██      ┃
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  ░░░░░  ┃   78    ┃  7.8%   ┃ █       ┃
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  █████  ┃   67    ┃  6.7%   ┃ █       ┃
┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  █████  ┃  █████  ┃  ░░░░░  ┃   13    ┃  1.3%   ┃         ┃
┗━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━

The same question as a correlation. ϕ is the correlation of two columns' missingness masks:
high and positive means the two go missing together.

In [5]:
missingcooccurrence(df)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃    ϕ    ┃  regi…  ┃  date   ┃   age   ┃  scho…  ┃  inco…  ┃   lab   ┃  weig…  ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  regi…  ┃    —    ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃
┃  date   ┃    ·    ┃    —    ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃
┃   age   ┃    ·    ┃    ·    ┃    —    ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃
┃  scho…  ┃    ·    ┃    ·    ┃    ·    ┃    —    ┃    ·    ┃    ·    ┃    ·    ┃
┃  inco…  ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃    —    ┃  -0.05  ┃  -0.02  ┃
┃   lab   ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃  -0.05  ┃    —    ┃  0.83   ┃
┃  weig…  ┃    ·    ┃    ·    ┃    ·    ┃    ·    ┃  -0.02  ┃  0.83   ┃    —    ┃
┗━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┛
 pairwise ϕ of missingness masks ┊ n = 1000 rows


`lab` and `weight` come out strongly associated, which is the skipped-visit mechanism showing
through. Nothing else pairs up, because nothing else was built to.

## Column by column

The sparkline is *where along the rows* the missing values sit — flat means spread evenly,
lumpy means the absence has a position, which usually means it has a cause.

In [6]:
missingsummary(df)

 column     type       missing        %  distribution
 lab        Float64        442   44.20%  ▄▄▃▄▄▄▄▅▄▅▃▄▅▃▃▃▄▄▅▄
 weight     Float64        351   35.10%  ▄▃▃▃▃▃▃▄▃▄▃▄▄▃▃▃▄▃▄▂
 income     Int64          202   20.20%  ▂▂▂▂▁▂▃▂▃▂▂▃▁▂▂▂▁▂▂▂
 region     String           0    0.00%                      
 date       Date             0    0.00%                      
 age        Int64            0    0.00%                      
 schooling  Int64            0    0.00%                      
 995 missing of 7000 cells (14.21%) across 7 columns ┊ bins of 50 rows


## What complete-case analysis costs

`missingrows` prices listwise deletion as the table stands: the `0` line is what survives
`dropmissing`, and everything under it is what you would be throwing away.

In [7]:
missingrows(df)

 missing/row  rows        %  distribution
 0             436   43.60%  ██████████████████████████████
 1             200   20.00%  ██████████████
 2             297   29.70%  ████████████████████
 3              67    6.70%  █████
 436 complete rows (43.60%) ┊ 564 with ≥1 missing (56.40%) ┊ 4 distinct counts across 7 columns


`missingdrop` prices the alternative — trading a variable for rows. It walks the greedy path,
at each step removing the column that turns the most rows complete, and flags the step that
maximizes the surviving complete-case block. Whether that trade is worth making is a modeling
judgment; the package only puts a number on it.

In [8]:
missingdrop(df)

 drop    cols  complete        %  distribution
 —          7       436   43.60%  █████████████
 income     6       558   55.80%  █████████████████
 lab        5       649   64.90%  ███████████████████
 weight     4      1000  100.00%  ██████████████████████████████  ◀ most complete-case cells
 436 of 1000 rows complete as given (43.60%) ┊ dropping 3 columns leaves 1000 complete across 4 columns (100.00%)


## Absence that does not say `missing`

`schooling` looked complete above, and it is not: `99` means "ignored". `isna` counts sentinels
as holes without rewriting the table.

Pass a single predicate and it applies to every column, which is almost never right — `99` is a
code in one column and a perfectly good number in another. A `NamedTuple` gives one predicate
per column, with `ismissing` assumed for the columns left out. Test `ismissing` first and let
`||` short-circuit: `missing == 99` is `missing`, not `false`, and would throw in a boolean
context.

In [9]:
missingsummary(df; isna = (schooling = x -> ismissing(x) || x == 99,))

 column     type       missing        %  distribution
 lab        Float64        442   44.20%  ▄▄▃▄▄▄▄▅▄▅▃▄▅▃▃▃▄▄▅▄
 weight     Float64        351   35.10%  ▄▃▃▃▃▃▃▄▃▄▃▄▄▃▃▃▄▃▄▂
 schooling  Int64          206   20.60%  ▂▁▂▃▃▂▂▂▂▂▃▂▂▃▃▂▂▁▁▃
 income     Int64          202   20.20%  ▂▂▂▂▁▂▃▂▃▂▂▃▁▂▂▂▁▂▂▂
 region     String           0    0.00%                      
 date       Date             0    0.00%                      
 age        Int64            0    0.00%                      
 1201 missing of 7000 cells (17.16%) across 7 columns ┊ bins of 50 rows


## Missingness against a grouping

The rows of the heatmap do not have to be blocks of table positions. `by` groups them by the
values of a column — a category, or a calendar period of a `Date` column — which turns the
vertical axis into something you can reason about.

By region, the income band is visible where the refusal was built in:

In [10]:
plotmissing(df; by = :region)

┏━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃     ┃    0%   ┃    0%   ┃    0%   ┃    0%   ┃   20%   ┃   44%   ┃   35%   ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃ row ┃  regi…  ┃  date   ┃   age   ┃  scho…  ┃  inco…  ┃   lab   ┃  weig…  ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃east ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┃north┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┃south┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┃west ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ▒▒▒▒▒  ┃  ▓▓▓▓▓  ┃  ▓▓▓▓▓  ┃
┗━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┛

MissingPatterns.Analysis: 1000 × 7 DataFrame
 Grouping:    by region → 4×7 cells
 Missing (count):             995               ┊ Missing (%):          14.21%
 Present (count):            6005               ┊ Present (

By year, the skipped visits fade as the study settles down:

In [11]:
plotmissing(df; by = :date, period = :year, order = :cluster)

┏━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃     ┃   44%   ┃   35%   ┃   20%   ┃    0%   ┃    0%   ┃    0%   ┃    0%   ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃ row ┃   lab   ┃  weig…  ┃  inco…  ┃  regi…  ┃  date   ┃   age   ┃  scho…  ┃
┣━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃2019 ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2020 ┃  █████  ┃  █████  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2021 ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2022 ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2023 ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┃2024 ┃  ▓▓▓▓▓  ┃  ▒▒▒▒▒  ┃  ▒▒▒▒▒  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃  ░░░░░  ┃
┗━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┻━━━━━━━━━┛

MissingPatterns.Analysis: 1000 × 7 DataFrame
 Grouping:    by d

## Getting the numbers out

Every view has a data counterpart that returns a Tables.jl row table instead of printing —
same kernels, so a number read here can never disagree with the one drawn above. No display
compression, no caps.

In [12]:
DataFrame(missingstats(df))

Row,column,eltype,nmissing,npresent,nrows,pct
,Symbol,Type,Int64,Int64,Int64,Float64
1,region,String,0,1000,1000,0.0
2,date,Date,0,1000,1000,0.0
3,age,Int64,0,1000,1000,0.0
4,schooling,Int64,0,1000,1000,0.0
5,income,"Union{Missing, Int64}",202,798,1000,20.2
6,lab,"Union{Missing, Float64}",442,558,1000,44.2
7,weight,"Union{Missing, Float64}",351,649,1000,35.1


In [13]:
# the most co-missing pairs, by ϕ
DataFrame(first(sort(missingpairstats(df); by = r -> -r.phi), 5))

Row,a,b,phi,jaccard,n11,n1,n2,nrows
,Symbol,Symbol,Float64,Float64,Int64,Int64,Int64,Int64
1,lab,weight,0.826299,0.794118,351,442,351,1000
2,income,weight,-0.0203626,0.13786,67,202,351,1000
3,income,lab,-0.0465618,0.141844,80,202,442,1000
4,region,date,NaN,NaN,0,0,0,1000
5,region,age,NaN,NaN,0,0,0,1000


## Auditing an imputation

`plotmissingdiff` compares two versions of the same table and marks where missing values were
resolved (`-`) or introduced (`+`). Filling `income` with its median resolves that column and
touches nothing else — which is the point of looking rather than trusting.

In [14]:
imputed = copy(df)
imputed.income = coalesce.(df.income, round(Int, median(skipmissing(df.income))))

plotmissingdiff(df, imputed; color = :never)

┏━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓
┃ re… +0% ┃ da… +0% ┃ age +0% ┃ sc… +0% ┃ i… -20% ┃ lab +0% ┃ we… +0% ┃
┣━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━╋━━━━━━━━━┫
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·····  ┃
┃  ·····  ┃  ·····  ┃  ·····  ┃  ·····  ┃  -----  ┃  ·····  ┃  ·

## Where to go next

- [Documentation](https://dantebertuzzi.github.io/MissingPatterns.jl/stable) — every keyword of
  every entry point
- [`obis-missingness.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/obis-missingness.ipynb)
  — the same diagnostics on real data: marine biodiversity records pulled live from OBIS with
  [OBISClient.jl](https://github.com/dantebertuzzi/OBISClient.jl)